# 10 · Tucker decomposition on real data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Descomposición de Tucker con datos reales** — Construir un tensor real de viajes en taxi, comprimir cada modo con HOSVD y explorar el compromiso entre tamaño, error e interpretación.

Build a real tensor from New York taxi trips, compress each mode with HOSVD, and explore the trade-off between size, reconstruction error, and interpretable structure.

## What you will be able to do

- Build and interpret a genuine order-3 tensor from a flat table of real trips.
- Unfold the tensor along each mode and explain what information each matricization exposes.
- Compute Tucker/HOSVD using only unfolding, SVD, and `einsum`.
- Change the rank of each mode independently and measure reconstruction error versus compression.
- Read the temporal factor matrix and connect a learned component back to real hourly taxi activity.

> 🇪🇸 **Al terminar podrás:** construir e interpretar un tensor real de orden 3; desplegarlo por cada modo; calcular Tucker/HOSVD con SVD y `einsum`; variar el rango de cada eje y medir error frente a compresión; e interpretar un factor temporal comparándolo con la actividad horaria real.

## Setup

Run this first. We use 6,433 real New York taxi trips and build a tensor indexed by pickup borough, dropoff borough, and hour of day.

> 🇪🇸 Ejecuta primero esta celda. Usaremos 6.433 viajes reales en taxi de Nueva York y construiremos un tensor indexado por distrito de origen, distrito de destino y hora del día.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

TAXIS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
taxis = pd.read_csv(TAXIS)

def unfold(T, axis):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

def hosvd_bases(T):
    return [
        np.linalg.svd(unfold(T, axis), full_matrices=False)[0]
        for axis in range(T.ndim)
    ]

taxis["pickup_dt"] = pd.to_datetime(taxis["pickup"], errors="coerce")
taxis["hour"] = taxis["pickup_dt"].dt.hour

sub = taxis.dropna(
    subset=["pickup_borough", "dropoff_borough", "hour"]
).copy()
sub["hour"] = sub["hour"].astype(int)

pickup_names = sorted(sub["pickup_borough"].unique())
dropoff_names = sorted(sub["dropoff_borough"].unique())

pickup_index = {name: i for i, name in enumerate(pickup_names)}
dropoff_index = {name: i for i, name in enumerate(dropoff_names)}

T = np.zeros((len(pickup_names), len(dropoff_names), 24), dtype=float)

for (p, d, h), count in sub.groupby(
    ["pickup_borough", "dropoff_borough", "hour"]
).size().items():
    T[pickup_index[p], dropoff_index[d], int(h)] = float(count)

print("taxi rows / filas:", len(taxis))
print("usable trips / viajes utilizables:", int(T.sum()))
print("tensor shape / forma:", T.shape)
print("pickup boroughs / origen:", pickup_names)
print("dropoff boroughs / destino:", dropoff_names)

taxi rows / filas: 6433
usable trips / viajes utilizables: 6383
tensor shape / forma: (4, 5, 24)
pickup boroughs / origen: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens']
dropoff boroughs / destino: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


## Why this matters

A matrix has two axes. A tensor can have three or more, and each axis can carry a different kind of meaning.

For our taxi tensor,

`T[pickup, dropoff, hour]`

stores a real trip count. The three modes answer different questions:

- **pickup mode:** which origins behave similarly?
- **dropoff mode:** which destinations behave similarly?
- **hour mode:** which times of day share similar traffic structure?

**HOSVD** applies an SVD to each unfolding and keeps the strongest directions for each mode. Tucker then combines those directions through a small **core tensor**.

This is better described as a multilinear extension of truncated SVD ideas across several tensor modes — not as “PCA itself becoming a tensor factorization.”

### How to use the folded solutions / Cómo usar las soluciones plegadas

Each exercise has a **Solution / Solución** cell that is intentionally closed. Try the `TODO` first; then open the solution.

> 🇪🇸 Un tensor conserva varios ejes con significados distintos. HOSVD aplica SVD a cada despliegue y conserva las direcciones dominantes de cada modo. Tucker combina esas direcciones mediante un tensor núcleo pequeño.
>
> Es más preciso entenderlo como una extensión multilineal de las ideas de SVD truncada a varios modos del tensor, no como “PCA convertido en una factorización tensorial”.
>
> Cada ejercicio tiene una celda **Solution / Solución** cerrada a propósito. Primero intenta el `TODO`; después abre la solución.

### Learning cycle: Predict → Run → Explain

Before each exercise, predict the shape, rank, or dominant pattern. Then run the computation and explain what the result means in the original taxi data.

> 🇪🇸 **Predice → Ejecuta → Explica:** anticipa la forma, el rango o el patrón dominante; luego ejecuta y traduce el resultado de vuelta al contexto de los viajes reales.

## Exercise 1 — read the tensor before decomposing it

Before compressing anything, understand what the entries mean.

For example, `T[i, j, h]` is the number of usable real trips that started in pickup borough `i`, ended in dropoff borough `j`, and were picked up during hour `h`.

### What should you try?

1. Print `T.shape` and `T.sum()`.
2. Find the busiest hour overall.
3. Unfold along each of the three modes and confirm that every unfolding contains the same total number of entries.
4. Move **Hour / Hora** to inspect the real origin→destination count matrix at different times.

> 🇪🇸 Antes de descomponer, entiende el tensor. Busca la hora con más viajes, revisa las tres formas de unfolding y usa el slider **Hour / Hora** para inspeccionar la matriz real origen→destino a distintas horas.

In [2]:
# TODO
# 1. Print T.shape and T.sum().
# 2. Compute T.sum(axis=(0, 1)) and find the busiest hour.
# 3. Print unfold(T, axis).shape for axis=0,1,2.
# 4. Predict which hours should show the most concentrated traffic.

In [3]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
print("T.shape:", T.shape)
print("total usable trips / viajes utilizables:", int(T.sum()))

by_hour = T.sum(axis=(0, 1))
busiest_hour = int(np.argmax(by_hour))

print("busiest hour / hora más ocupada:", busiest_hour)
print("trips at busiest hour / viajes:", int(by_hour[busiest_hour]))

for axis in range(3):
    M = unfold(T, axis)
    print(
        f"axis/eje {axis}: shape/forma={M.shape} | "
        f"same entries/mismas entradas={M.size == T.size}"
    )

hour_slider = widgets.IntSlider(
    value=busiest_hour,
    min=0,
    max=23,
    step=1,
    description="Hour / Hora:",
    continuous_update=False,
    style={"description_width": "90px"},
)

def show_hour(hour):
    matrix = T[:, :, hour]

    print(
        f"hour/hora={hour} | total trips/viajes={int(matrix.sum())}"
    )

    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    im = ax.imshow(matrix, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(dropoff_names)))
    ax.set_xticklabels(dropoff_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pickup_names)))
    ax.set_yticklabels(pickup_names, fontsize=8)
    ax.set_xlabel("dropoff borough / destino")
    ax.set_ylabel("pickup borough / origen")
    ax.set_title(f"Real taxi counts at hour {hour} / Viajes reales a la hora {hour}")
    fig.colorbar(im, ax=ax, label="trip count / viajes")
    plt.tight_layout()
    plt.show()

hour_output = widgets.interactive_output(
    show_hour,
    {"hour": hour_slider},
)

display(widgets.VBox([hour_slider, hour_output]))

T.shape: (4, 5, 24)
total usable trips / viajes utilizables: 6383
busiest hour / hora más ocupada: 18
trips at busiest hour / viajes: 417
axis/eje 0: shape/forma=(4, 120) | same entries/mismas entradas=True
axis/eje 1: shape/forma=(5, 96) | same entries/mismas entradas=True
axis/eje 2: shape/forma=(24, 20) | same entries/mismas entradas=True


## Exercise 2 — HOSVD: compress each mode separately

Now compute one SVD basis per unfolding.

If the retained ranks are `(r₁, r₂, r₃)`, then:

- `U₁` summarizes pickup patterns;
- `U₂` summarizes dropoff patterns;
- `U₃` summarizes hourly patterns;
- the core has shape `(r₁, r₂, r₃)`.

The core is produced by contracting all three modes at once:

`core = einsum('ijk,ia,jb,kc->abc', T, U1, U2, U3)`

and reconstruction reverses that contraction.

### What should you try?

1. Compute the full SVD basis of each unfolding.
2. Start with ranks `(2, 2, 3)`.
3. Build the core with one `einsum`.
4. Reconstruct with one `einsum`.
5. Compute relative Frobenius error and compression ratio.
6. Change the three rank sliders independently and inspect how error and storage trade off.

> 🇪🇸 Ahora cada modo obtiene su propia base SVD. El tensor núcleo resume cómo interactúan esos factores. Cambia los tres rangos de manera independiente y observa que reducir almacenamiento siempre tiene un costo de reconstrucción.

In [4]:
# TODO
# 1. Compute one SVD basis for each unfolding.
# 2. Keep ranks (2, 2, 3).
# 3. Build the Tucker core with one np.einsum call.
# 4. Reconstruct T with one np.einsum call.
# 5. Compute relative error and compression ratio.

In [5]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
bases = hosvd_bases(T)

def tucker_from_ranks(T, bases, ranks):
    factors = [
        bases[axis][:, :ranks[axis]]
        for axis in range(3)
    ]

    core = np.einsum(
        "ijk,ia,jb,kc->abc",
        T,
        factors[0],
        factors[1],
        factors[2],
    )

    recon = np.einsum(
        "abc,ia,jb,kc->ijk",
        core,
        factors[0],
        factors[1],
        factors[2],
    )

    error = np.linalg.norm(T - recon) / np.linalg.norm(T)

    compressed_numbers = (
        core.size
        + sum(factor.size for factor in factors)
    )
    compression = T.size / compressed_numbers

    return core, factors, recon, error, compression

default_ranks = (
    min(2, T.shape[0]),
    min(2, T.shape[1]),
    min(3, T.shape[2]),
)

core, factors, recon, error, compression = tucker_from_ranks(
    T,
    bases,
    default_ranks,
)

print("default ranks / rangos:", default_ranks)
print("factor shapes / formas:", [u.shape for u in factors])
print("core shape / forma núcleo:", core.shape)
print("relative error / error relativo:", f"{error:.4f}")
print("compression / compresión:", f"{compression:.2f}x")

pickup_rank = widgets.IntSlider(
    value=default_ranks[0],
    min=1,
    max=T.shape[0],
    step=1,
    description="Pickup rank:",
    continuous_update=False,
    style={"description_width": "100px"},
)

dropoff_rank = widgets.IntSlider(
    value=default_ranks[1],
    min=1,
    max=T.shape[1],
    step=1,
    description="Dropoff rank:",
    continuous_update=False,
    style={"description_width": "110px"},
)

hour_rank = widgets.IntSlider(
    value=default_ranks[2],
    min=1,
    max=min(12, T.shape[2]),
    step=1,
    description="Hour rank:",
    continuous_update=False,
    style={"description_width": "95px"},
)

compare_hour = widgets.IntSlider(
    value=busiest_hour,
    min=0,
    max=23,
    step=1,
    description="Hour / Hora:",
    continuous_update=False,
    style={"description_width": "90px"},
)

def explore_tucker(r_pickup, r_dropoff, r_hour, hour):
    ranks = (r_pickup, r_dropoff, r_hour)

    core, factors, recon, error, compression = tucker_from_ranks(
        T,
        bases,
        ranks,
    )

    print(
        f"ranks/rangos={ranks} | core/núcleo={core.shape} | "
        f"error={error:.4f} | compression/compresión={compression:.2f}x"
    )

    vmax = max(T[:, :, hour].max(), recon[:, :, hour].max())

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.3))

    axes[0].imshow(
        T[:, :, hour],
        cmap="viridis",
        aspect="auto",
        vmin=0,
        vmax=vmax,
    )
    axes[0].set_title(f"real / real — hour {hour}")

    axes[1].imshow(
        recon[:, :, hour],
        cmap="viridis",
        aspect="auto",
        vmin=0,
        vmax=vmax,
    )
    axes[1].set_title(f"Tucker reconstruction / reconstrucción")

    for ax in axes:
        ax.set_xticks(range(len(dropoff_names)))
        ax.set_xticklabels(dropoff_names, rotation=45, ha="right", fontsize=7)
        ax.set_yticks(range(len(pickup_names)))
        ax.set_yticklabels(pickup_names, fontsize=7)

    plt.tight_layout()
    plt.show()

tucker_output = widgets.interactive_output(
    explore_tucker,
    {
        "r_pickup": pickup_rank,
        "r_dropoff": dropoff_rank,
        "r_hour": hour_rank,
        "hour": compare_hour,
    },
)

display(
    widgets.VBox([
        widgets.HTML(
            "<b>Tucker rank explorer / Explorador de rangos Tucker:</b> "
            "change each mode independently. / "
            "cambia cada modo de manera independiente."
        ),
        pickup_rank,
        dropoff_rank,
        hour_rank,
        compare_hour,
        tucker_output,
    ])
)

default ranks / rangos: (2, 2, 3)
factor shapes / formas: [(4, 2), (5, 2), (24, 3)]
core shape / forma núcleo: (2, 2, 3)
relative error / error relativo: 0.0670
compression / compresión: 4.71x


## Exercise 3 — what did the temporal factor learn?

Compression is useful, but interpretation is where Tucker becomes more than a storage trick.

The hour factor matrix has one row for each hour and one column for each retained temporal component. Singular-vector signs are arbitrary, so we compare magnitudes when asking where a component is strongest.

### What should you try?

1. Inspect the first temporal factor.
2. Find the hour where its absolute loading is largest.
3. Compare that hour with the busiest hour in the raw taxi counts.
4. Move **Component / Componente** to inspect other temporal patterns.
5. Explain why a factor can capture a pattern even though nobody explicitly labeled “rush hour.”

> 🇪🇸 La matriz de factores temporales tiene una fila por hora. Busca dónde alcanza mayor magnitud cada componente y compáralo con los conteos horarios reales. El signo de un vector singular es arbitrario, así que interpreta principalmente la forma y la magnitud del patrón.

In [6]:
# TODO
# 1. Use the hour-mode SVD basis.
# 2. Inspect the first temporal component.
# 3. Find its peak absolute loading.
# 4. Compare it with the busiest hour in T.sum(axis=(0,1)).
# 5. Repeat for another component and describe the pattern.

In [7]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
hour_basis = bases[2]
raw_hour_counts = T.sum(axis=(0, 1))

first_component = hour_basis[:, 0]
first_peak = int(np.argmax(np.abs(first_component)))

print("first component peak / pico primer componente:", first_peak)
print("raw busiest hour / hora real más ocupada:", busiest_hour)

component_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=min(6, hour_basis.shape[1]),
    step=1,
    description="Component / Componente:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_hour_factor(component):
    idx = component - 1
    factor = hour_basis[:, idx]

    peak = int(np.argmax(np.abs(factor)))

    raw_scaled = raw_hour_counts / raw_hour_counts.max()
    factor_scaled = np.abs(factor)
    factor_scaled = factor_scaled / factor_scaled.max()

    print(
        f"component/componente={component} | "
        f"peak absolute loading / pico absoluto={peak}"
    )
    print(
        f"raw busiest hour / hora real más ocupada={busiest_hour}"
    )

    fig, ax = plt.subplots(figsize=(6.6, 3.2))
    ax.plot(
        range(24),
        raw_scaled,
        marker="o",
        label="raw hourly trips / viajes reales",
    )
    ax.plot(
        range(24),
        factor_scaled,
        marker="o",
        label=f"|temporal factor {component}| / |factor temporal|",
    )
    ax.axvline(
        peak,
        linestyle="--",
        linewidth=1,
        label=f"factor peak / pico={peak}",
    )
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("hour / hora")
    ax.set_ylabel("scaled magnitude / magnitud escalada")
    ax.set_title("Raw activity vs learned temporal factor / Actividad real vs factor")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

factor_output = widgets.interactive_output(
    explore_hour_factor,
    {"component": component_slider},
)

display(widgets.VBox([component_slider, factor_output]))

first component peak / pico primer componente: 18
raw busiest hour / hora real más ocupada: 18


## What just happened

You built a complete Tucker/HOSVD pipeline from real observations.

1. **Real tensor construction:** a flat taxi table became an order-3 tensor `pickup × dropoff × hour`.
2. **Unfolding:** each mode exposed a different matrix view without losing any entries.
3. **HOSVD:** SVD supplied one low-dimensional basis per mode.
4. **Tucker core:** one `einsum` contracted all three axes into a smaller core, and another reconstructed the tensor.
5. **Rank trade-off:** changing pickup, dropoff, and hour ranks independently changed both storage and reconstruction error.
6. **Interpretation:** the hour factor exposed temporal structure that could be compared directly with real hourly trip counts.

### The sentence to remember

> **Tucker compresses a tensor by learning a basis for each mode and a small core that tells those mode-specific patterns how to interact.**

This is why tensor decompositions are useful in domains such as imaging, recommender systems, neuroscience, and multi-condition biological measurements: the axes represent genuinely different kinds of structure.

> 🇪🇸 Construiste Tucker/HOSVD de extremo a extremo con datos reales. Cada modo obtuvo su propia base, el núcleo resumió sus interacciones y los sliders mostraron que la compresión y el error dependen de cuánto rango conservas en cada eje.
>
> **Frase para recordar:** Tucker comprime un tensor aprendiendo una base para cada modo y un núcleo pequeño que describe cómo interactúan esos patrones específicos de cada eje.

---

## Time for Kahoot 🎯

**Kahoot 3 — Convolution & Tensor Decompositions** · 6 questions, about 5 minutes.

> 🇪🇸 **Convolución y descomposiciones tensoriales** — 6 preguntas, unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-3)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_3_convolution_decompositions.xlsx)

Next up: **11 · Wrap-up and take-homes** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)